# Build Coupled Wflow-SFINCS Model


## Stage Contract
Requires: Location config, restored source modules, and the upstream notebook artifacts for this stage.
Produces: a build coupled model artifacts for the Greensboro inland Wflow/SFINCS workflow.
Next: Run the next numbered Greensboro flood notebook after the readiness table is clean.


In [ ]:
from os.path import join
from copy import deepcopy
import os
from pathlib import Path
import sys

os.environ.pop("DEBUG", None)
from study_location import bootstrap
session = bootstrap()
location_root = session.location_root
repo_root = session.repo_root
for module_name in list(sys.modules):
    if module_name == "study_location" or module_name.startswith(("design_events", "sfincs_runs", "wflow_runs")):
        sys.modules.pop(module_name, None)

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from hydromt_sfincs import DATADIR, SfincsModel

# build domains, source points, observations, and physics checks.
import sfincs_runs.build_base.inland_base as inland_sfincs
from sfincs_runs.build_base import (
    plan_inland_sfincs_base,
    plan_inland_sfincs_domain_set,
    plot_sfincs_handoff_basemap,
    sfincs_rivers_inflow_geoms,
    set_observations,
    validate_physics,
    write_inland_sfincs_domain_set_manifest,
)
# create native submodels and verify coupling inputs.
from wflow_runs import (
    coupled_domain_review,
    wflow_artifact_inventory,
    wflow_handoff_contract,
    build_wflow_data_catalog,
    build_wflow_submodel,
    validate_wflow_reservoir_staticmaps,
    plot_wflow_basemap,
    plot_wflow_ldd_components,
    wflow_catalog_source_readiness,
    write_wflow_domain_set_manifest,
)
# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime, domain_summary

def configread(path):
    with open(path, encoding="utf-8") as handle:
        return yaml.safe_load(handle)

In [ ]:
# Load this Location Workspace.
runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=True)
location_root = runtime.location_root
repo_root = runtime.repo_root
location_name = runtime.location_name
config = runtime.config
paths = runtime.design_paths

def location_path(value):
    return runtime.resolve_location_path(value)

sfincs_data_catalog = location_root / "data/static/data_catalogue.yaml"
wflow_data_catalog = runtime.wflow_base_root.parent / "data_catalog.yml"
wflow_base_root = runtime.wflow_base_root
wflow_network_path = Path(paths["usgs_streamgage_network_geojson"])

## Rerun Control


In [ ]:
rerun = True

## Part 1 — Coupled Domain Plans

Wflow and SFINCS use different geometry meanings: Wflow is hydrologic and can extend upstream; SFINCS is hydraulic coverage around each SMART-DS component.


### Step 1 · Wflow watershed and SFINCS coverage plans

Plan Wflow from the configured hydrologic boundary (`boundary_handoff_watershed`) and plan SFINCS from the selected SMART-DS coverage box. The first Wflow build intentionally ignores stale SFINCS handoff artifacts so built LDD and Wflow-native river geometry can drive source placement.


In [ ]:
wflow_first_config = deepcopy(config)
wflow_first_config.setdefault("wflow", {}).setdefault("domain_set", {})["ignore_sfincs_handoff_artifacts"] = True
wflow_build_plan, wflow_domain_plan, _ = domain_summary(wflow_first_config, location_root)
sfincs_base_plan = plan_inland_sfincs_base(config, paths)
sfincs_domain_plan = plan_inland_sfincs_domain_set(config, paths)

if wflow_domain_plan.status != "ready":
    raise RuntimeError(f"Candidate Wflow Domain Set requires review before Wflow build: {wflow_domain_plan.status}: {wflow_domain_plan.issues}")
if sfincs_domain_plan.status != "ready":
    raise RuntimeError(f"SFINCS coverage domains require review before build: {sfincs_domain_plan.status}: {sfincs_domain_plan.issues}")

sfincs_domain_manifest = write_inland_sfincs_domain_set_manifest(sfincs_domain_plan, config, paths)
wflow_domain_manifest = write_wflow_domain_set_manifest(wflow_domain_plan, wflow_first_config, paths)
domain_set = yaml.safe_load(location_path(config["sfincs_domain_set"]["domain_manifest"]).read_text())
sfincs_domains = list(domain_set["domains"])
selected_submodels = list(wflow_domain_plan.submodels)

selected_source_subregion_ids
exposure_subregion_id


### Step 2 · Hydrologic boundary and handoff map

Plot the Wflow watershed boundary, selected SMART-DS coverage, and Wflow-led stream/SFINCS-boundary handoff points before building coupled artifacts. The Wflow domain-set plan is the authority for stream-boundary source IDs; SFINCS writes source locations from that plan, not from reviewed USGS gage IDs.


In [ ]:
domain_review = coupled_domain_review(
    wflow_first_config,
    location_root,
    sfincs_domains=sfincs_domains,
    wflow_domain_plan=wflow_domain_plan,
    figsize=(9, 7),
)

sfincs_domain_gdf = domain_review.sfincs_domain_gdf
plt.show()

## Part 2 — HydroMT-Wflow Native Build

Check the HydroMT-Wflow inputs and build recipe, then build the hydrologic submodels from the Wflow domain-set stream-boundary handoff plan.


### Step 3 · HydroMT-Wflow data catalog readiness

HydroMT-Wflow needs the local DEM-derived hydrography basemap, landcover, soil parameter maps, and event forcing sources before the submodels can be built.


In [ ]:
wflow_catalog_path = build_wflow_data_catalog(config, paths)
wflow_source_readiness = pd.DataFrame(wflow_catalog_source_readiness(wflow_catalog_path))
missing_required_wflow_source = (
    wflow_source_readiness["required_for_build"].fillna(False).astype(bool)
    & wflow_source_readiness["local_file"].fillna(False).astype(bool)
    & ~wflow_source_readiness["exists"].fillna(False).astype(bool)
)
required_wflow_sources_missing = wflow_source_readiness[missing_required_wflow_source]

if not required_wflow_sources_missing.empty:
    missing_lines = [
        f"{row.source}: {row.uri}"
        for row in required_wflow_sources_missing.itertuples(index=False)
    ]
    display(wflow_source_readiness)
    raise FileNotFoundError(
        "Missing required HydroMT-Wflow source files before build:\n"
        + "\n".join(missing_lines)
        + f"\nRun locations/{location_name}/02_flood/01_region_setup.ipynb through the Terrain, Landcover, and Wflow static inputs cell first. "
        + "For a fresh Wflow static pull, launch that notebook with FLOOD_RM_FETCH_DEM=1; landcover fetch is enabled by default."
    )

wflow_source_readiness

### Step 4 · Wflow build steps

Inspect the HydroMT-Wflow workflow. `build_wflow_submodel` keeps this syntax but replaces the template region with each reviewed `subbasin + uparea` outlet region.


In [ ]:
wf_config = configread(str(location_path(config["wflow"]["build_config"])))
print(wf_config.keys())
pd.DataFrame({"step": [next(iter(step)) for step in wf_config.get("steps", [])]})

### Step 5 · Build or reuse Wflow submodels before SFINCS handoff placement

Build Wflow first from the boundary-handoff watershed plan to produce coherent LDD, stream order, and native river geometry. SFINCS handoff sources are derived from those built Wflow rivers in the next part.


In [ ]:
# First Wflow build: ignore any stale SFINCS handoff source artifacts so HydroMT-Wflow
# builds the hydrologic domain from the configured boundary-handoff subbasin plan.
wflow_native_build_summary = [
    build_wflow_submodel(
        wflow_first_config,
        paths,
        submodel_id=submodel["wflow_submodel_id"],
        force=rerun,
        write_catalog=False,
    )
    for submodel in selected_submodels
]

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_native_build_summary}
for summary in wflow_native_build_summary:
    fig, axes = plot_wflow_ldd_components(summary["model"])
    fig.savefig(wflow_base_root / summary["wflow_submodel_id"] / "wflow_ldd_components.png", dpi=300, bbox_inches="tight")

pd.DataFrame([
    {k: v for k, v in summary.items() if k != "model"}
    for summary in wflow_native_build_summary
])

## Part 3 — HydroMT-SFINCS Coverage Build

SFINCS domains are hydraulic coverage boxes around SMART-DS components. They are not assumed to be full watersheds.


### Initialize SFINCS coverage models

Each SMART-DS coverage box gets its own `SfincsModel` root. The discharge sources are added after Wflow-native rivers are available.


In [ ]:
sfincs_model_rows = [
    {
        "sfincs_domain_id": domain["sfincs_domain_id"],
        "region": str(location_path(domain["region"]).relative_to(repo_root)),
        "base_model_root": str(location_path(domain["base_model_root"]).relative_to(repo_root)),
        "handoff_source_count": len(domain["handoff_source_ids"]),
        "grid_resolution_m": float(config["sfincs"]["grid_resolution_m"]),
    }
    for domain in sfincs_domains
]

pd.DataFrame(sfincs_model_rows)

### SFINCS build configuration

Use the standard HydroMT-SFINCS build recipe for the grid, elevation, masks, and subgrid. In stream-boundary mode, Wflow coupling sources are then created from built Wflow river/SFINCS-boundary intersections.


In [ ]:
sf_config_path = location_path(config["sfincs"]["build_config"])
sf_recipe = (config.get("_model_recipes") or {}).get("sfincs_build")
if sf_recipe is not None:
    sf_config_path.parent.mkdir(parents=True, exist_ok=True)
    sf_config_path.write_text(
        "# GENERATED FILE — do not edit. Overwritten when the sfincs_build model YAML extraction runs.\n"
        "# Source of truth is the location config and the code that produces this file.\n"
        + yaml.safe_dump(sf_recipe, sort_keys=False),
        encoding="utf-8",
    )
sf_config = configread(str(sf_config_path))
sf_config["setup_grid_from_region"]["res"] = config["sfincs"]["grid_resolution_m"]
sf_config["setup_grid_from_region"]["crs"] = config["sfincs"].get("model_crs", config["project"]["model_crs"])
sf_config["setup_dep"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_dep"] = [{"elevtn": "dem_region"}]
sf_config["setup_subgrid"]["datasets_rgh"] = [{"lulc": "landcover_region"}]
sf_config.pop("setup_river_outflow", None)
print(sf_config.keys())


### Step 11 · Build or reuse SFINCS grids, masks, subgrid, and Wflow source points

The source GeoJSON is written after both the SFINCS grid and the first Wflow build exist, using built Wflow river/SFINCS-boundary intersections filtered by stream order. Wflow then uses those SFINCS `src` points as its handoff gauges.


In [ ]:
sfincs_build_report = inland_sfincs.build_domains(
    config,
    paths,
    force=rerun,
)

# Built Wflow river/SFINCS-boundary intersections are the source of truth for
# handoff count and coordinates in stream-boundary mode. Refresh both manifests
# before the final Wflow build so Wflow gauges are rebuilt at those source IDs.
sfincs_domain_plan = plan_inland_sfincs_domain_set(config, paths)
if sfincs_domain_plan.status != "ready":
    raise RuntimeError(f"SFINCS coverage domains require review after built-river handoff build: {sfincs_domain_plan.status}: {sfincs_domain_plan.issues}")
sfincs_domain_manifest = write_inland_sfincs_domain_set_manifest(sfincs_domain_plan, config, paths)
domain_set = yaml.safe_load(location_path(config["sfincs_domain_set"]["domain_manifest"]).read_text())
sfincs_domains = list(domain_set["domains"])

wflow_build_plan, wflow_domain_plan, _ = domain_summary(config, location_root)
if wflow_domain_plan.status != "ready":
    raise RuntimeError(f"Wflow Domain Set requires review after built-river handoff build: {wflow_domain_plan.status}: {wflow_domain_plan.issues}")
wflow_domain_manifest = write_wflow_domain_set_manifest(wflow_domain_plan, config, paths)
selected_submodels = list(wflow_domain_plan.submodels)

sfincs_models = {}
sfincs_build_rows = []
handoff_layers = []
sfincs_build_by_domain = (
    sfincs_build_report.set_index("sfincs_domain_id").to_dict("index")
    if not sfincs_build_report.empty and "sfincs_domain_id" in sfincs_build_report
    else {}
)

for domain in sfincs_domains:
    domain_id = domain["sfincs_domain_id"]
    root = location_path(domain["base_model_root"])
    validate_physics(root, config)

    sf = SfincsModel(
        root=str(root),
        mode="r+",
        data_libs=[str(sfincs_data_catalog), str(wflow_data_catalog)],
    )
    sf.read()
    sfincs_models[domain_id] = sf

    handoff_locations = root / "gis/wflow_handoff_sources.geojson"
    src = gpd.read_file(handoff_locations)
    rivers = sfincs_rivers_inflow_geoms(sf)
    handoff_layers.append(src)
    build_row = sfincs_build_by_domain.get(domain_id, {})
    sfincs_build_rows.append({
        "sfincs_domain_id": domain_id,
        "status": build_row.get("status", "review_required"),
        "built": bool(build_row.get("built", False)),
        "grid_resolution_m": float(config["sfincs"]["grid_resolution_m"]),
        "rivers_inflow_features": int(len(rivers)),
        "handoff_source_count": int(len(src)),
        "root": str(root.relative_to(repo_root)),
    })

pd.DataFrame(sfincs_build_rows)

### Step 9 · SFINCS coverage QA plots

Plot each SFINCS coverage grid with reviewed gages, HydroMT-SFINCS native `rivers_inflow` linework, and Wflow->SFINCS source points.


In [ ]:
handoff_gdf = pd.concat(handoff_layers, ignore_index=True)
handoff_gdf = gpd.GeoDataFrame(handoff_gdf, geometry="geometry", crs=handoff_layers[0].crs)

from hydromt_sfincs.utils import get_bounds_vector

sfincs_coupling_qa = []
for domain in sfincs_domains:
    sfincs_domain_id = domain["sfincs_domain_id"]
    sf = sfincs_models[sfincs_domain_id]
    sf.elevation.data["dep"].attrs.update(long_name="elevation", units="m")
    handoff_locations = location_path(domain["base_model_root"]) / "gis/wflow_handoff_sources.geojson"
    rivers = sfincs_rivers_inflow_geoms(sf)
    observations = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=sf.crs)
    if wflow_network_path.exists():
        observations = set_observations(sf, wflow_network_path)

    # plot_bounds draws the mask=3 outflow boundary. In the coupled Wflow->SFINCS
    # model these are distinct from src points, which are Wflow inflows at stream
    # and coverage-box crossings.
    fig, ax, plot_qa = plot_sfincs_handoff_basemap(
        sf,
        handoff_sources=handoff_locations,
        rivers=rivers,
        observations=observations,
        config=config,
        paths=paths,
        domain_region=domain["region"],
        figsize=(8, 6),
    )
    fig.savefig(join(sf.root.path, "sfincs_basemap.png"), dpi=450, bbox_inches="tight")

    # Coupling QA: confirm both boundary types exist and the inflow sources are not sitting
    # on the outflow boundary (a ~0 m distance would mean conflicting boundary conditions).
    src_pts = sf.discharge_points.gdf
    mask = sf.grid.data["mask"]
    outflow_bnd = get_bounds_vector(mask)
    outflow_bnd = outflow_bnd[outflow_bnd["value"] == 3]
    if not outflow_bnd.empty and not src_pts.empty:
        outflow_line = outflow_bnd.to_crs(src_pts.crs).geometry.union_all()
        min_src_to_outflow_m = round(float(src_pts.geometry.distance(outflow_line).min()), 1)
    else:
        min_src_to_outflow_m = "no outflow bnd / no src"
    sfincs_coupling_qa.append({
        "sfincs_domain_id": sfincs_domain_id,
        **plot_qa,
        "outflow_boundary_cells_mask3": int((mask == 3).sum()),
        "min_src_to_outflow_distance_m": min_src_to_outflow_m,
    })

pd.DataFrame(sfincs_coupling_qa)


### Step 13 · SFINCS handoff source table

Review the source coordinates and their river-intersection provenance before rebuilding Wflow gauges.


In [ ]:
handoff_columns = [
    "sfincs_domain_id",
    "sfincs_handoff_id",
    "wflow_submodel_id",
    "site_no",
    "handoff_placement",
    "handoff_location_review_status",
    "stream_boundary_river_source",
    "stream_boundary_candidate_count",
]
handoff_columns = [column for column in handoff_columns if column in handoff_gdf.columns]
handoff_gdf[handoff_columns]


## Part 4 — Coupling Back Into Wflow

After SFINCS writes boundary source points, Wflow is rebuilt with gauge outputs at exactly those source points. Each Wflow hydrologic domain may be much larger than its SFINCS coverage box.


### Step 14 · Rebuild Wflow gauges at SFINCS boundary sources

This final Wflow build uses the refreshed boundary-handoff subbasin plan from the built-river SFINCS source GeoJSONs. The `setup_gauges` SFINCS layer reads exactly those boundary-source points.



In [ ]:
reservoirs_enabled = bool(config.get("collection", {}).get("national_hydrography", {}).get("reservoirs", {}).get("enabled", False))
reservoir_preflight_rows = []
if reservoirs_enabled:
    for submodel in selected_submodels:
        submodel_id = submodel["wflow_submodel_id"]
        model_root = wflow_base_root / submodel_id
        staticmaps_path = model_root / "staticmaps.nc"
        if staticmaps_path.exists():
            report = validate_wflow_reservoir_staticmaps(model_root, required=True, raise_on_error=False)
            report.insert(0, "wflow_submodel_id", submodel_id)
            reservoir_preflight_rows.append(report)
    reservoir_preflight = pd.concat(reservoir_preflight_rows, ignore_index=True) if reservoir_preflight_rows else pd.DataFrame()
    display(reservoir_preflight)
    if not reservoir_preflight.empty and reservoir_preflight["status"].isin(["failed", "review_required"]).any() and not rerun:
        raise RuntimeError("Reservoirs are enabled, but the existing Wflow base is stale. Set rerun=True and rerun this step.")

wflow_build_summary = [
    build_wflow_submodel(
        config,
        paths,
        submodel_id=submodel["wflow_submodel_id"],
        force=rerun,
        write_catalog=False,
    )
    for submodel in selected_submodels
]

wflow_models = {summary["wflow_submodel_id"]: summary["model"] for summary in wflow_build_summary}
pd.DataFrame([{k: v for k, v in summary.items() if k != "model"} for summary in wflow_build_summary])

### Step 15 · Final coupled Wflow basemaps

Plot the native Wflow model with filled Wflow basin and SFINCS coverage areas, Wflow-native river vectors, intended SFINCS handoff sources, and the model's post-snap `gauges_sfincs` layer. This is the main check that the upstream Wflow model feeds the right hydraulic coverage box boundaries without confusing source artifacts with snapped Wflow gauge cells.


In [ ]:
sfincs_background_elevation = [sf.elevation.data["dep"] for sf in sfincs_models.values()]
for submodel_id, wf in wflow_models.items():
    observation_gages_path = wflow_base_root.parent / "domain_set_gauges" / f"{submodel_id}_observation_gauges.geojson"
    observation_gages = gpd.read_file(observation_gages_path) if observation_gages_path.exists() else None
    fig, ax = plot_wflow_basemap(
        wf,
        gages=observation_gages,
        sfincs_domains=sfincs_domain_gdf,
        background_elevation=sfincs_background_elevation,
        figsize=(8, 6),
        title=f"{submodel_id} coupled Wflow base map",
    )
    fig.savefig(wflow_base_root / submodel_id / "wflow_basemap.png", dpi=450, bbox_inches="tight")


### Step 16 · Final Wflow gauge QA

List the Wflow gauge layers that will produce discharge for SFINCS and verify each SFINCS source maps to one nearby Wflow gauge. The LDD/staticmaps are not replotted here because gauge insertion does not change them.


In [ ]:
handoff_review = wflow_handoff_contract(
    config,
    location_root,
    wflow_models=wflow_models,
    handoff_gdf=handoff_gdf,
    wflow_base_root=wflow_base_root,
)

handoff_contract = handoff_review.handoff_contract
display(handoff_contract)
handoff_review.gauge_layers
